# Module 11: Fine-tuning language models responsibly

Perform real parameter-efficient gradient updates on annotated messages, preserve the base model, and audit held-out loss and replay.

Use the complete source folder and the documented isolated CPU framework environment. Run in order, retain named text/tables, save, restart, and run all. First source/model downloads need internet; valid caches support replay. No earlier notebook state or paid account is needed.

The public script calls the same implementation. Small-model mechanics and valid software execution do not establish task quality or accessible notebook-interface operation.


In [ ]:
from pathlib import Path
import sys
import json
import tempfile
import platform
import importlib.metadata
import numpy as np
import pandas as pd

# Find shared code from the complete checkout/archive, not copied notebook logic.
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "ml_ai_it"]
ROOT = next((p for p in candidates if (p / "src/practical_ml_ai").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete ml_ai_it source folder.")
sys.path.insert(0, str(ROOT / "src"))
from practical_ml_ai.course_runner import run_module

# A new directory allows clean replay without silently replacing frozen evidence.
(ROOT / "runs").mkdir(exist_ok=True)
parent = Path(tempfile.mkdtemp(prefix="module11-notebook-", dir=ROOT / "runs"))
run_dir = parent / "study"
print("Evidence will be saved to:", run_dir)


## Inspect the real source and boundaries

Train rank-8 PEFT LoRA on q_proj and v_proj in every attention layer, using 256 real training messages, batches of 8 and a 200-step maximum. Prompt and padding labels are masked. Check validation completion loss every 25 steps, retaining the best checkpoint including step zero. Validation generation uses 40 messages; the predeclared balanced final comparison uses 100 disjoint messages. Save safetensors and verify reloaded logits.

Before running, predict the kinds of errors and the relevant denominators. The source data and human labels remain unchanged.

In [ ]:
from practical_ml_ai.continuation_data import sms_splits
split, source_manifest = sms_splits(limit=None)
for name, part in split.items():
    # Display IDs/counts rather than raw messages that may contain personal data.
    print(name, "rows:", len(part), "positive labels:", int(part.label.sum()))
print("First training text IDs:", split["train"].text_id.head(3).tolist())
print("Sampling rule:", source_manifest["sampling"])


## Inspect masking and adaptation

The rank-2 construction below exposes the matrix update. The actual study uses rank 8 in every query/value layer. The cached base is cloned before PEFT wraps its modules, so later studies still use the unchanged base model.

In [ ]:
from practical_ml_ai.language import local_model, packed_example
from practical_ml_ai.continuation_data import balanced_real_subset
model, tokenizer, revision = local_model()
real_training_row = balanced_real_subset(split["train"], 24).iloc[0]
input_tokens, loss_labels = packed_example(tokenizer, real_training_row)
print("Actual training text ID:", real_training_row.text_id)
print("Token shape:", tuple(input_tokens.shape))
print("Masked prompt positions:", int((loss_labels == -100).sum()))
print("Supervised completion positions:", int((loss_labels != -100).sum()))
# Do not train on validation/final labels or accidentally unmask the whole prompt.


## Guided construction before independent practice

If W has output-by-input shape (6,4), it has 24 parameters. A rank-2 A has shape (2,4), B has shape (6,2), and BA has shape (6,4). Their 20 parameters are fewer here; savings become much larger for wide projections. Starting B at zero makes the initial update zero. A frozen base participates in forward computation without optimizer updates. Labels -100 exclude prompt positions from the completion-loss objective; they are a library masking convention.

Predict the output before running. Then complete the module guided page's independent modification.


In [ ]:
import numpy as np
learning_w = np.ones((6,4))
learning_a = np.ones((2,4)) * .02
learning_b = np.zeros((6,2))
print("base",learning_w.shape,"A",learning_a.shape,"B",learning_b.shape)
assert (learning_b @ learning_a).shape == learning_w.shape
assert np.allclose(learning_w + learning_b @ learning_a,learning_w)
print("base parameters",learning_w.size,"adapter parameters",learning_a.size+learning_b.size)

# Hand-written rank-2 construction; this is separate from the full PEFT study.
import torch
from practical_ml_ai.language import low_rank_query
base_projection = torch.nn.Linear(4, 6)
base_projection.requires_grad_(False)
small_adapter = low_rank_query(base_projection, rank=2)
probe = torch.ones(1, 4)
torch.testing.assert_close(small_adapter(probe), base_projection(probe))
print("Initially zero low-rank update; base projection remains frozen.")

## Run the trained adapter study

The default 200-step CPU study takes substantially longer than the earlier eight-step construction. Predict whether completion loss, format compliance and label accuracy must improve together. Checkpoint selection uses validation loss, including the option to retain the zero-step adapter.

In [ ]:
# Execute the declared default protocol; final labels do not select settings.
result = run_module(11, run_dir)
print((run_dir / "summary.txt").read_text(encoding="utf-8"))

## Inspect the matched evidence

Read training_history.csv, validation_generation.csv and generation_comparison.csv. The first describes optimization, the second development behavior and the third the final comparison. Keep these purposes separate. Check that the selected step has the lowest measured validation loss before interpreting final results.

In [ ]:
table = pd.read_csv(run_dir / "training_history.csv")
print("Evidence table: training_history.csv")
print(table.head(15).to_string(index=False))
print("Complete report:", run_dir / "index.html")


## Verify one interpretation

Use the actual output to check a denominator, fitting boundary, update, or budget. A passing software check is separate from whether the method is useful.


In [ ]:
from safetensors.torch import load_file
adapter_state = load_file(run_dir / "adapter/adapter_model.safetensors")
print("Saved adapter tensors:", len(adapter_state))
print("One tensor shape:", tuple(next(iter(adapter_state.values())).shape))
print("Selected step:", result["selected_step"])
print("Base/adapted JSON validity:", result["base_schema_rate"], result["adapted_schema_rate"])
print("Base/adapted label accuracy:", result["base_generation_accuracy"], result["adapted_generation_accuracy"])
assert result["adapter_replay_verified"]

## Record versions and save

Retain the source/protocol manifest, output directory, commands, versions, and your own interpretation. Complete the lesson's practice and reflection; optional depth is skippable. Restart-and-run-all makes hidden state visible, but repeated public final data are not fresh independent evaluation.


In [ ]:
versions = {"python": platform.python_version()}
for package in ["numpy","pandas","torch","transformers","peft","accelerate"]:
    versions[package] = importlib.metadata.version(package)
(run_dir / "environment.json").write_text(json.dumps(versions, indent=2) + "\n", encoding="utf-8")
print("Reproduction versions:", versions)
print("Your saved evidence:", run_dir)


## Independent practice

Identify trainable parameters and masked token positions. Verify a nonzero adapter update, exact source/revision, disjoint IDs, held-out losses, and saved-adapter replay. Explain why a lower token loss does not prove safe, factual, or accurate task output.

Optional: Use a predeclared validation plan for another rank, step budget, or target projection. More complete supervised training and a credible prospective evaluation require additional real data, compute, and review; they are optional depth.

Explain one limitation and a next action. If using GenAI, verify its suggestions against actual code/tables and follow local disclosure rules. Instructors can adapt the linked practice criteria while preserving learning goals and accessibility.
